# 01 · Data Generation

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)

print("Working folder:", os.getcwd())

In [ ]:
!pip install python-sat -q

In [ ]:
import random
import pickle
from pysat.solvers import Minisat22

## Settings

In [ ]:
SEED = 42
TRAIN_PAIRS = 10000
VAL_PAIRS = 1000
TEST_PAIRS = 1000
MIN_VARS = 10
MAX_VARS = 40
TEST_LARGE_VARS = [40, 60, 80]
TEST_LARGE_PAIRS = 200

random.seed(SEED)

## Random clause

In [ ]:
def clause_size():
    k = 2
    if random.random() < 0.3:
        k += 1
    while random.random() > 0.4:
        k += 1
    return k


def random_clause(n_vars):
    k = min(clause_size(), n_vars)
    variables = random.sample(range(1, n_vars + 1), k)
    return [v if random.random() < 0.5 else -v for v in variables]

## SR(n) pair

In [ ]:
def is_sat(clauses):
    with Minisat22(bootstrap_with=clauses) as solver:
        return solver.solve()


def generate_pair(n_vars):
    clauses = []
    solver = Minisat22()

    while True:
        clause = random_clause(n_vars)
        solver.add_clause(clause)
        if solver.solve():
            clauses.append(clause)
        else:
            break

    solver.delete()

    unsat_clauses = clauses + [clause]

    flipped = clause.copy()
    i = random.randrange(len(flipped))
    flipped[i] = -flipped[i]
    sat_clauses = clauses + [flipped]

    return sat_clauses, unsat_clauses

## Quick check

In [ ]:
sat_clauses, unsat_clauses = generate_pair(10)

print("Clauses:", len(sat_clauses))
print("Last clause SAT version:  ", sat_clauses[-1])
print("Last clause UNSAT version:", unsat_clauses[-1])
print("SAT version is satisfiable:  ", is_sat(sat_clauses))
print("UNSAT version is satisfiable:", is_sat(unsat_clauses))

## Dataset

In [ ]:
def make_dataset(n_pairs, min_vars, max_vars):
    data = []
    for _ in range(n_pairs):
        n_vars = random.randint(min_vars, max_vars)
        sat_clauses, unsat_clauses = generate_pair(n_vars)
        data.append({"n_vars": n_vars, "clauses": sat_clauses, "label": 1})
        data.append({"n_vars": n_vars, "clauses": unsat_clauses, "label": 0})
    return data

In [ ]:
train_data = make_dataset(TRAIN_PAIRS, MIN_VARS, MAX_VARS)
val_data = make_dataset(VAL_PAIRS, MIN_VARS, MAX_VARS)
test_data = make_dataset(TEST_PAIRS, MIN_VARS, MAX_VARS)

large_test_data = {}
for n in TEST_LARGE_VARS:
    large_test_data[n] = make_dataset(TEST_LARGE_PAIRS, n, n)

print("Train:", len(train_data))
print("Val:  ", len(val_data))
print("Test: ", len(test_data))
for n in TEST_LARGE_VARS:
    print(f"Test n={n}:", len(large_test_data[n]))

## Sanity check

In [ ]:
sample = random.sample(test_data, 200)
correct = sum(is_sat(p["clauses"]) == bool(p["label"]) for p in sample)
print(f"Labels correct: {correct}/{len(sample)}")

## Statistics

In [ ]:
def stats(data):
    n_clauses = [len(p["clauses"]) for p in data]
    n_vars = [p["n_vars"] for p in data]
    sat_share = sum(p["label"] for p in data) / len(data)
    print(f"Variables: {min(n_vars)}–{max(n_vars)}")
    print(f"Clauses:   {min(n_clauses)}–{max(n_clauses)}, mean {sum(n_clauses) / len(n_clauses):.1f}")
    print(f"SAT share: {sat_share:.2f}")

stats(train_data)

## Save

In [ ]:
with open("train.pkl", "wb") as f:
    pickle.dump(train_data, f)

with open("val.pkl", "wb") as f:
    pickle.dump(val_data, f)

with open("test.pkl", "wb") as f:
    pickle.dump(test_data, f)

with open("test_large.pkl", "wb") as f:
    pickle.dump(large_test_data, f)

print("Saved.")